<a href="https://colab.research.google.com/github/dhanvarsh29/Bus/blob/main/aqi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving intervention_dataset_500rows.xlsx to intervention_dataset_500rows.xlsx


In [ ]:
import os
os.listdir()


['.config', 'sample_data']

In [ ]:
import pandas as pd

# Load Excel file
data = pd.read_excel("intervention_dataset_500rows.xlsx")

print("Dataset Loaded Successfully!")
print(data.head())
print("\nDataset Shape:", data.shape)


Dataset Loaded Successfully!
   AQI  traffic  stubble  industry  dust  wind_speed  humidity  odd_even  \
0  352     0.38     0.49      0.18  0.19        3.39        55         0   
1  520     0.34     0.54      0.17  0.19        3.94        70         0   
2  356     0.18     0.38      0.22  0.20        1.51        81         0   
3  321     0.18     0.26      0.08  0.22        3.86        49         1   
4  438     0.36     0.30      0.06  0.25        1.86        71         1   

   crackers_ban  construction_halt  
0             1                  1  
1             1                  0  
2             1                  0  
3             0                  0  
4             1                  1  

Dataset Shape: (500, 10)


In [ ]:
import numpy as np

print("\nFixing Dataset: Making interventions reduce AQI...\n")

# Save original AQI
data["AQI_base"] = data["AQI"]

# Apply reduction rule
data["AQI"] = data["AQI_base"] \
              - (data["odd_even"] * 25) \
              - (data["crackers_ban"] * 15) \
              - (data["construction_halt"] * 30)

# Remove temporary column
data = data.drop("AQI_base", axis=1)

print("Dataset Updated Successfully!")
print(data.head())



Fixing Dataset: Making interventions reduce AQI...

Dataset Updated Successfully!
   AQI  traffic  stubble  industry  dust  wind_speed  humidity  odd_even  \
0  307     0.38     0.49      0.18  0.19        3.39        55         0   
1  505     0.34     0.54      0.17  0.19        3.94        70         0   
2  341     0.18     0.38      0.22  0.20        1.51        81         0   
3  296     0.18     0.26      0.08  0.22        3.86        49         1   
4  368     0.36     0.30      0.06  0.25        1.86        71         1   

   crackers_ban  construction_halt  
0             1                  1  
1             1                  0  
2             1                  0  
3             0                  0  
4             1                  1  


In [ ]:
# Input Features
X = data.drop("AQI", axis=1)

# Output Target
y = data["AQI"]

print("\nFeatures Shape:", X.shape)
print("Target Shape:", y.shape)



Features Shape: (500, 9)
Target Shape: (500,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("\nTraining Data:", X_train.shape)
print("Testing Data:", X_test.shape)



Training Data: (400, 9)
Testing Data: (100, 9)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

print("\nTraining Intervention Model...\n")

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

print("Intervention Model Trained Successfully!")



Training Intervention Model...

Intervention Model Trained Successfully!


In [ ]:
from sklearn.metrics import mean_absolute_error

# Predict on test set
y_pred = model.predict(X_test)

# Calculate MAE
mae = mean_absolute_error(y_test, y_pred)

print("\nModel Evaluation Done!")
print("Mean Absolute Error (MAE):", mae)



Model Evaluation Done!
Mean Absolute Error (MAE): 70.65225000000001


In [ ]:
print("\nSimulating Policy Interventions...\n")

# Example current conditions (change anytime)
current_conditions = {
    "traffic": 0.35,
    "stubble": 0.45,
    "industry": 0.10,
    "dust": 0.10,
    "wind_speed": 1.5,
    "humidity": 70,
}

# Policy Scenarios
policies = {
    "No Policy": [0, 0, 0],
    "Odd-Even": [1, 0, 0],
    "Crackers Ban": [0, 1, 0],
    "Construction Halt": [0, 0, 1],
    "All Policies": [1, 1, 1],
}

results = {}

# Predict AQI for each policy
for policy_name, policy_values in policies.items():

    input_data = [
        current_conditions["traffic"],
        current_conditions["stubble"],
        current_conditions["industry"],
        current_conditions["dust"],
        current_conditions["wind_speed"],
        current_conditions["humidity"],
        policy_values[0],  # odd_even
        policy_values[1],  # crackers_ban
        policy_values[2],  # construction_halt
    ]

    predicted_aqi = model.predict([input_data])[0]
    results[policy_name] = predicted_aqi

# Print Results
print("Predicted AQI Under Different Policies:\n")

for k, v in results.items():
    print(f"{k}: AQI = {v:.2f}")



Simulating Policy Interventions...

Predicted AQI Under Different Policies:

No Policy: AQI = 401.39
Odd-Even: AQI = 397.54
Crackers Ban: AQI = 400.12
Construction Halt: AQI = 402.10
All Policies: AQI = 374.44


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [ ]:
# Best policy = minimum AQI
best_policy = min(results, key=results.get)

print("\nBest Policy Recommendation:", best_policy)
print("Expected AQI After Policy:", results[best_policy])



Best Policy Recommendation: All Policies
Expected AQI After Policy: 374.445


In [ ]:
import joblib

joblib.dump(model, "intervention_model.joblib")
print("Model saved successfully!")


Model saved successfully!


In [ ]:
from google.colab import files
files.download("intervention_model.joblib")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predict on test data
y_pred = model.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("📌 Model Performance Metrics:\n")
print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("R² Score:", r2)


📌 Model Performance Metrics:

Mean Absolute Error (MAE): 70.65225000000001
Mean Squared Error (MSE): 7012.3214342500005
Root Mean Squared Error (RMSE): 83.73960493249297
R² Score: 0.016082934310017505


In [ ]:
print("\nGenerating AI Recommendation...\n")

traffic = current_conditions["traffic"]
stubble = current_conditions["stubble"]

if stubble > 0.40:
    recommendation = "High stubble burning detected → Focus on farm-fire control in Punjab/Haryana."
elif traffic > 0.30:
    recommendation = "Traffic emissions rising → Implement Odd-Even and congestion restrictions."
else:
    recommendation = "Industrial contribution high → Tighten factory emission regulations."

print("AI Recommendation:")
print(recommendation)



Generating AI Recommendation...

AI Recommendation:
High stubble burning detected → Focus on farm-fire control in Punjab/Haryana.


In [ ]:
[x for x in globals() if "model" in x.lower()]


[]